In [ ]:
%load_ext autoreload
%autoreload 2

# coding

> Representing library sequences and motifs as integers, so that scoring a motif is arithmetic
> rather than string matching.

In [ ]:
#| default_exp coding

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_fail
import pandas as pd   # the worked examples only; the module itself needs no pandas

In [ ]:
#| export
import functools
import numpy as np
from collections import defaultdict

## Why the scan is arithmetic

The scan in `00_core` scores **every** motif of every pattern against every sequence, which is where
its runtime goes. On strings — a Python set of motif strings per sequence — that costs roughly 10 s
and 2 GB of RAM per pattern on a 30k-sequence library; on integers it is well under a second.

A motif of pattern `(d1, spacer, d2)` is pinned down entirely by its `d1 + d2` defined bases — the
spacer is `N` by construction — so it can be held as an integer in `[0, 4**(d1+d2))`. Every
per-motif statistic then becomes an `np.bincount` over those integers, and the library is encoded
once and reused for every pattern.

Nothing here decides anything biological: its only job is to give exactly the answer a readable
string-based scan would, faster. The checks in this notebook hold the encoding to that, and
`tests/test_reference_scan.py` holds the whole scan to it.

### The base alphabet, and why it is `ACGT` in that order

`A=0, C=1, G=2, T=3`, and anything else — `N`, lowercase junk, the padding of a short
sequence — is `4`.

The order is not arbitrary. Complementary bases sit at opposite ends of it, so **complementing
a base is `3 - code`**: A↔T is 0↔3, C↔G is 1↔2. That is the whole reverse-complement
implementation, and it is why `encode_library` can return the reverse strand for the price of
one subtraction and one reversal.

Code `4` does double duty: it marks a base that is not ACGT *and* the padding that squares off a
ragged library into a rectangular array. Both must be excluded from the scan, and because they
share a marker, one test excludes both — there is no separate length bookkeeping to get wrong.

In [ ]:
#| export
_BASE_CODES = np.full(256, 4, dtype=np.uint8)  # 4 = anything that is not A/C/G/T
for _i, _b in enumerate("ACGT"):
    _BASE_CODES[ord(_b)] = _BASE_CODES[ord(_b.lower())] = _i
_CODE_BASES = np.frombuffer(b"ACGT", dtype=np.uint8)

def encode_library(
    sequences:list, # DNA strings, e.g. inserts with their flanks; lengths may differ
)->tuple: # `(fwd, rc)`: uint8 base codes (n_sequences, max_length) of each sequence and its rc
    "Encode DNA sequences as base codes, the form the scan reads (`encoded_library`)."
    max_len = max(len(s) for s in sequences)
    fwd = np.full((len(sequences), max_len), 4, dtype=np.uint8)
    rc = np.full_like(fwd, 4)
    for i, s in enumerate(sequences):
        codes = _BASE_CODES[np.frombuffer(s.encode(), dtype=np.uint8)]
        fwd[i, :len(s)] = codes
        rc[i, :len(s)] = np.where(codes < 4, 3 - codes, 4)[::-1]  # A<->T, C<->G
    return fwd, rc

In [ ]:
fwd, rc = encode_library(["ACGTT", "ggN"])
fwd, rc

(array([[0, 1, 2, 3, 3],
        [2, 2, 4, 4, 4]], dtype=uint8),
 array([[0, 0, 1, 2, 3],
        [4, 1, 1, 4, 4]], dtype=uint8))

In [ ]:
#| hide
test_eq(fwd, np.array([[0, 1, 2, 3, 3], [2, 2, 4, 4, 4]]))
test_eq(rc,  np.array([[0, 0, 1, 2, 3], [4, 1, 1, 4, 4]]))  # AACGT and NCC, padded

Both strands are left-aligned and padded on the right, so row `i` of `rc` is the reverse
complement of sequence `i` occupying its first `len(s)` columns — not a reversal of the padded
row, which would put the padding at the front.

In [ ]:
#| hide
import random

In [ ]:
#| hide
# revcomp really is the string operation, and applying it twice is the identity
_COMP = str.maketrans("ACGT", "TGCA")
_B = {b: i for i, b in enumerate("ACGT")}
_rng = random.Random(0)
for _ in range(200):
    s = "".join(_rng.choice("ACGTN") for _ in range(_rng.randint(5, 40)))
    f, r = encode_library([s])
    test_eq(f[0, :len(s)].tolist(), [_B.get(c, 4) for c in s])
    test_eq(r[0, :len(s)].tolist(), [_B.get(c, 4) for c in s.translate(_COMP)[::-1]])
    test_eq(encode_library([s.translate(_COMP)[::-1]])[1][0, :len(s)].tolist(),
            f[0, :len(s)].tolist())          # rc(rc(s)) == s

### From a window to an integer, and back

Write the `d1 + d2` defined bases of a motif as the digits of a **base-4 numeral**:

    CAC NNNN GTAT   ->  digits C,A,C,G,T,A,T  ->  1·4⁶ + 0·4⁵ + 1·4⁴ + 2·4³ + 3·4² + 0·4 + 3

`_pattern_window_codes` builds that numeral by Horner's method — `code = code*4 + next_digit`,
one pass over the defined positions in increasing order. The trick that makes it fast is that
each step operates on a **sliding view**, `encoded_seqs[:, j:j+n_windows]`: position `j` of the
window starting at `w` is absolute index `w+j`, so as `w` runs over all windows that column is
exactly the slice above. Every window of every sequence is therefore built in parallel, with one
vector operation per defined base rather than one per window.

**Rejecting windows.** A window containing a non-ACGT base anywhere — spacer included, to match
the `[ATGC]` regex the reference implementation uses — is set to `4**(d1+d2)`, one past the
largest real code. Which windows those are comes from a prefix sum over the `== 4` mask: the
number of bad bases in `[w, w+width)` is `n_bad[w+width] - n_bad[w]`, so all windows are tested
at once. Padding is code 4, so a window running off the end of a short sequence is rejected by
the same test, with nothing extra written for it.

**Decoding** reverses the numeral. Base 4 is two bits, so digit `i` counted from the most
significant is `(code >> 2*(n-1-i)) & 3` — the same digits in the same order the encoder wrote
them. `_decode_motif_codes` then drops `N` into the spacer positions.

In [ ]:
#| export
def _pattern_window_codes(encoded_seqs, pattern):
    """
    Motif code of every window of every sequence for `pattern` (d1, spacer, d2).
    Returns an int32 array of shape (n_sequences, n_windows); windows containing a
    base coded 4 (non-ACGT or padding) get the out-of-range code 4**(d1+d2).
    """
    d1, spacer, d2 = pattern
    width = d1 + spacer + d2
    # A motif code is a base-4 numeral with d1+d2 digits, and the rejection sentinel is one
    # past the largest of them, so the whole scheme has to fit in the array's dtype. int32
    # holds 15 defined bases; the shipped patterns use at most 8. Checked rather than assumed:
    # on numpy >= 2 the sentinel assignment below happens to raise, but on numpy 1.x the same
    # assignment wraps silently and every count for this pattern would be quietly wrong.
    if 4 ** (d1 + d2) > np.iinfo(np.int32).max:
        raise ValueError(
            f"pattern {pattern} has {d1 + d2} defined bases; the motif code would overflow "
            f"int32 (max 15). Such a pattern is not usable anyway: it would need "
            f"{4 ** (d1 + d2):.3g} counting bins.")
    n_windows = max(encoded_seqs.shape[1] - width + 1, 0)   # a pattern wider than the sequences has none
    codes = np.zeros((encoded_seqs.shape[0], n_windows), dtype=np.int32)
    for j in list(range(d1)) + list(range(d1 + spacer, width)):  # defined positions only
        codes *= 4
        codes += encoded_seqs[:, j:j + n_windows]
    # Like the [ATGC] regex, reject windows with a non-ACGT base anywhere, spacer included.
    n_bad = np.zeros((encoded_seqs.shape[0], encoded_seqs.shape[1] + 1), dtype=np.int32)
    np.cumsum(encoded_seqs == 4, axis=1, out=n_bad[:, 1:])
    codes[n_bad[:, width:] != n_bad[:, :n_windows]] = 4 ** (d1 + d2)
    return codes

def _decode_motif_codes(codes, pattern):
    """
    Converts integer motif codes of `pattern` (d1, spacer, d2) back to motif strings
    with 'N' in the spacer, e.g. CACNNNNGTAT for pattern (3, 4, 4).
    """
    d1, spacer, d2 = pattern
    n_defined = d1 + d2
    codes = np.asarray(codes, dtype=np.int64)
    out = np.full((len(codes), d1 + spacer + d2), ord("N"), dtype=np.uint8)
    bases = _CODE_BASES[(codes[:, None] >> (2 * np.arange(n_defined - 1, -1, -1))) & 3]
    out[:, :d1], out[:, d1 + spacer:] = bases[:, :d1], bases[:, d1:]
    return out.view(f"S{out.shape[1]}").ravel().astype(str).tolist()

With pattern `(1, 2, 2)`, `CAAGT` has a single window whose defined bases are C, G, T, i.e. code `1*16 + 2*4 + 3 = 27`. A window touching the `N` is rejected (code `4**3 = 64`):

In [ ]:
fwd, rc = encode_library(["CAAGT", "CAAGTN"])

# A=0 C=1 G=2 T=3, so a motif's code is literally its defined bases as a base-4 numeral:
# CNNGT of pattern (1,2,2) has defined bases C,G,T -> digits 1,2,3 -> int("123", 4).
CNNGT = int("123", 4)
REJECTED = 4 ** 3            # the sentinel: one past the largest 3-digit base-4 numeral
print(f"CNNGT = {CNNGT}, REJECTED = {REJECTED}")
print(_pattern_window_codes(fwd, (1, 2, 2)))
_decode_motif_codes([CNNGT, int("000", 4)], (1, 2, 2))

CNNGT = 27, REJECTED = 64
[[27 64]
 [27 64]]


['CNNGT', 'ANNAA']

In [ ]:
#| hide
test_eq((CNNGT, REJECTED), (27, 64))

test_eq(_pattern_window_codes(fwd, (1, 2, 2)), np.array([[CNNGT, REJECTED],
                                                        [CNNGT, REJECTED]]))
test_eq(_decode_motif_codes([CNNGT, int("000", 4)], (1, 2, 2)), ["CNNGT", "ANNAA"])

# A pattern wider than the sequences has no windows rather than failing (the numpy path; the
# numba kernel already handled it, which is why only a numba-free install hit this).
test_eq(_pattern_window_codes(encode_library(["ACGT", "TTTT"])[0], (4, 4, 4)).shape, (2, 0))

### What guarantees this is right?

The scan reports a motif string, but everything between the library and that string is integer
arithmetic. Four properties have to hold, or the output is confidently wrong rather than
obviously broken.

**1. Distinct motifs get distinct codes.** A base-4 numeral with a *fixed* number of digits is
unique to its digit string — that is just positional notation — and the digit count is fixed at
`d1+d2` for a given pattern. Two different motifs of the same pattern therefore cannot collide.

**2. Decoding is exactly the inverse of encoding.** Both walk the same `d1+d2` positions in the
same order; the encoder multiplies up, the decoder shifts down. Cheap enough to check
*exhaustively* rather than argue about, which this notebook's tests do.

**3. A rejected window can never be read as a motif.** Real codes fill `[0, 4**(d1+d2))` and the
sentinel is `4**(d1+d2)` exactly, so the single test `code < 4**(d1+d2)` separates them with no
gap and no overlap. This is why the sentinel is that value and not, say, `-1`.

**4. The arithmetic stays in range.** Codes are `int32`, which holds 15 defined bases; the
shipped patterns use at most 8. Beyond 15 the numeral would wrap and every count for that
pattern would be quietly wrong, so `_pattern_window_codes` raises instead.

Properties 1–3 hold by construction and are verified exhaustively by this notebook's tests. Whether the *whole scan*
means the same thing as a readable implementation is a separate question:
`tests/test_reference_scan.py` runs `_get_pattern_scores` against the string-based scan in
`tests/reference_scan.py` and requires identical tables.

In [ ]:
#| hide
# 1 & 2: exhaustive round-trip over the entire code space of several patterns.
for _pat in [(1, 0, 1), (6, 0, 0), (3, 4, 4), (4, 8, 4)]:
    _d1, _sp, _d2 = _pat
    _n = 4 ** (_d1 + _d2)
    _strs = _decode_motif_codes(np.arange(_n), _pat)
    test_eq(len(set(_strs)), _n)                                   # 1. injective
    for _s in _strs:                                               # spacer is always N
        test_eq(_s[_d1:_d1 + _sp], "N" * _sp)
    _back = np.zeros(_n, dtype=np.int64)                           # 2. re-encode by hand
    for _pos in list(range(_d1)) + list(range(_d1 + _sp, _d1 + _sp + _d2)):
        _back = _back * 4 + np.array([_B[_s[_pos]] for _s in _strs])
    test_eq(_back.tolist(), list(range(_n)))
print("round-trip exact over every code of", [(1, 0, 1), (6, 0, 0), (3, 4, 4), (4, 8, 4)])

# 3: the sentinel sits exactly one past the top real code, and only bad windows get it.
for _pat in [(6, 0, 0), (3, 4, 4)]:
    _w = _pat[0] + _pat[1] + _pat[2]
    test_eq(int(_pattern_window_codes(encode_library(["N" * _w])[0], _pat)[0, 0]),
            4 ** (_pat[0] + _pat[2]))
    test_eq(int(_pattern_window_codes(encode_library(["A" * _w])[0], _pat)[0, 0]), 0)

# 4: out-of-range patterns are refused rather than wrapped.
_fwd = encode_library(["ACGT" * 20])[0]
_pattern_window_codes(_fwd, (15, 0, 0))                            # fits
test_fail(lambda: _pattern_window_codes(_fwd, (16, 0, 0)), contains="overflow")

round-trip exact over every code of [(1, 0, 1), (6, 0, 0), (3, 4, 4), (4, 8, 4)]


### Encoding a motif back to its code

`_encode_motif` is the inverse of `_decode_motif_codes`: it turns a motif string back into the
integer the scan counts under. It returns `None` for anything it cannot represent — an
IUPAC-degenerate motif has no single code, and nor has a motif that does not fit the pattern — so
a caller can expand a degenerate motif into its instances (`_motif_instances` in `core`) or fall
back to a regex, rather than silently mis-assign sequences.
`tests/test_vectorized_presence.py` checks the two paths agree element for element.

In [ ]:
#| export
def _encode_motif(motif, pattern):
    """Integer code of `motif` under `pattern`, inverse of `_decode_motif_codes`.

    Returns None if the motif does not fit the pattern or contains a base outside ACGT --
    an IUPAC-degenerate motif has no single code (`core` reads it through its instances).
    """
    d1, spacer, d2 = pattern
    if len(motif) != d1 + spacer + d2:
        return None
    defined = motif[:d1] + motif[d1 + spacer:]
    if d1 + spacer + d2 and any(c != "N" for c in motif[d1:d1 + spacer]):
        return None                       # spacer must be undefined
    code = 0
    for ch in defined:
        v = "ACGT".find(ch)
        if v < 0:
            return None
        code = code * 4 + v
    return code

In [ ]:
# the decode example above, inverted: CNNGT is digits 1,2,3; an IUPAC base has no single code
_encode_motif("CNNGT", (1, 2, 2)), _encode_motif("YNNGT", (1, 2, 2))

(27, None)

In [ ]:
#| hide
test_eq(_encode_motif("CNNGT", (1, 2, 2)), int("123", 4))
# what has no single code returns None, so the caller knows to read it another way
test_eq(_encode_motif("YNNGT", (1, 2, 2)), None)          # IUPAC base
test_eq(_encode_motif("CNNG", (1, 2, 2)), None)           # wrong length for the pattern
test_eq(_encode_motif("CAAGT", (1, 2, 2)), None)          # spacer must be undefined

In [ ]:
#| hide
# exact inverse of _decode_motif_codes over the whole code space of a small pattern
for _c in range(4 ** 3):
    test_eq(_encode_motif(_decode_motif_codes([_c], (1, 2, 2))[0], (1, 2, 2)), _c)

### The two strands of one site

An enzyme cuts a double-stranded site, so `GAGACC` and `GGTCTC` are one motif seen from two
sides, and the scan must not count them as two. *Where* they can be folded together is decided by
the pattern, because a motif of `(d1, spacer, d2)` reverse-complements into one of
`(d2, spacer, d1)`:

- when that is the **same** shape — a contiguous motif (`spacer == 0`), or a balanced one
  (`d1 == d2`) — both spellings are codes in the *same* space, and one physical site is found
  twice, once per strand. These fold.
- otherwise the rc spelling lives in **another pattern's** space. Nothing to fold: scanning
  `(3, 4, 4)` on both strands already finds each site exactly once, because a `(4, 4, 3)` site
  is read on the reverse strand and spelled as `(3, 4, 4)`. This is why
  `find_restricted_motifs` rewrites every pattern to its `_canonical_pattern` and drops
  duplicates — searching both shapes would re-find the same sites in mirror spelling, at twice
  the cost.

Folding an asymmetric pattern would merge two *unrelated* motifs, since reversing the digits of a `(4, 4, 3)` code reads the `d2` bases where
the `d1` bases belong. `_rc_code_table` therefore refuses any pattern that is not self-rc.

The table itself is one observation: a code is the defined bases as a base-4 numeral, and the
defined bases of the reverse complement are those digits **reversed and complemented**
(`A<->T, C<->G` is `3 - digit`). The spacer is undefined on both strands, so it plays no part.
Because each base is two bits, both operations are bit arithmetic on the code: complementing
every digit is a single XOR with `11` repeated `k` times, and reversing the digits is reversing
the order of the 2-bit groups. `_rc_code_table` builds the table that way, and this notebook's tests
check it against `revcomp` on the motif strings over entire code spaces.

In [ ]:
#| export
def _pattern_is_self_rc(pattern):
    """Does a motif of `pattern` reverse-complement into another motif of the same pattern?

    A motif of `(d1, spacer, d2)` has its reverse complement in `(d2, spacer, d1)`, so the two
    spellings share a code space only when that swap changes nothing: a contiguous motif
    (`spacer == 0`) or a balanced one (`d1 == d2`). This is a property of the shape alone, so it
    holds for any pattern a caller supplies, not just the ones `get_patterns()` returns.
    """
    d1, spacer, d2 = pattern
    return spacer == 0 or d1 == d2


@functools.lru_cache(maxsize=None)
def _rc_code_table(pattern):
    """`table[code]` is the code of the reverse-complement motif, for a self-rc `pattern`.

    A code is the motif's defined bases read as a base-4 numeral, and the defined bases of the
    reverse complement are those digits reversed and complemented -- with `A=0 C=1 G=2 T=3`,
    complementing a base is `3 - code`. The spacer is undefined on both strands and plays no
    part.

    Raises for a pattern whose reverse complement has a different shape: there the rc motif is
    not in this code space at all, and folding on this table would merge unrelated motifs.
    """
    if not _pattern_is_self_rc(pattern):
        raise ValueError(
            f"pattern {pattern} is not self-reverse-complementary: a motif of it reverse-"
            f"complements into pattern {(pattern[2], pattern[1], pattern[0])}, which is a "
            f"different code space. There is nothing to fold.")
    k = pattern[0] + pattern[2]
    # Each base is a 2-bit group (A=00 C=01 G=10 T=11), so both halves of "reverse complement"
    # are bit operations on the code itself. Reverse: swap the 2-bit groups end for end, done
    # here as a fixed reversal of all sixteen groups of a 32-bit word followed by a shift that
    # drops the 16 - k unused ones. Complement: 3 - d for every digit, which on two bits is
    # d XOR 11, so the whole code complements as one XOR with k groups of 11.
    x = np.arange(4 ** k, dtype=np.uint32)
    x = ((x >> 2) & 0x33333333) | ((x & 0x33333333) << 2)    # swap neighbouring groups
    x = ((x >> 4) & 0x0F0F0F0F) | ((x & 0x0F0F0F0F) << 4)    # swap neighbouring pairs
    x = ((x >> 8) & 0x00FF00FF) | ((x & 0x00FF00FF) << 8)    # swap neighbouring bytes
    x = (x >> 16) | (x << 16)                                # swap the two halves
    table = ((x >> np.uint32(32 - 2 * k)) ^ np.uint32(4 ** k - 1)).astype(np.int64)
    # A lookup into this, rather than evaluating the formula on every window, because in numpy
    # each line above is a full pass over the array: on the 30k x 360 window-code matrix of
    # the 150 nt example library (`counts_panel.csv.gz`) the formula costs about twice the lookup (85 ms vs 45 ms per pattern).
    # The table is 4**k entries, the same size as the bincount the scan already allocates.
    table.setflags(write=False)           # cached and shared; never mutate in place
    return table

In [ ]:
# GAGACC and GGTCTC are one site: the table maps each spelling's code to the other's
_t = _rc_code_table((6, 0, 0))
print(_decode_motif_codes([_t[_encode_motif("GAGACC", (6, 0, 0))]], (6, 0, 0)))
# contiguous and balanced patterns fold; (3, 4, 4) does not, its rc motifs are (4, 4, 3)
[_pattern_is_self_rc(p) for p in [(6, 0, 0), (4, 4, 4), (3, 4, 4)]]

['GGTCTC']


[True, True, False]

In [ ]:
#| hide
test_eq(_decode_motif_codes([_t[_encode_motif("GAGACC", (6, 0, 0))]], (6, 0, 0)), ["GGTCTC"])
# a palindrome (EcoRI's GAATTC) is its own reverse complement, so it maps to itself
test_eq(_t[_encode_motif("GAATTC", (6, 0, 0))], _encode_motif("GAATTC", (6, 0, 0)))
test_eq([_pattern_is_self_rc(p) for p in [(6, 0, 0), (4, 4, 4), (3, 4, 4)]], [True, True, False])

In [ ]:
#| hide
from randseq.utils import revcomp

In [ ]:
#| hide
# the shape rule
test_eq(_pattern_is_self_rc((6, 0, 0)), True)     # contiguous: rc of a 6-mer is a 6-mer
test_eq(_pattern_is_self_rc((4, 4, 4)), True)     # balanced
test_eq(_pattern_is_self_rc((4, 4, 3)), False)    # rc has shape (3, 4, 4)
test_fail(lambda: _rc_code_table((4, 4, 3)), contains="nothing to fold")

# the table agrees with revcomp on the string, over whole code spaces
for _pat in [(1, 0, 1), (5, 0, 0), (6, 0, 0), (3, 4, 3), (4, 2, 4)]:
    _n = 4 ** (_pat[0] + _pat[2])
    _strs = _decode_motif_codes(np.arange(_n), _pat)
    _tbl = _rc_code_table(_pat)
    test_eq([revcomp(_s) for _s in _strs], list(_decode_motif_codes(_tbl, _pat)))
    test_eq(_tbl[_tbl].tolist(), list(range(_n)))        # rc is an involution
print("rc table == revcomp over every code of", [(1, 0, 1), (5, 0, 0), (6, 0, 0), (3, 4, 3), (4, 2, 4)])

# folding is exactly 'half the space, plus the palindromes'
for _pat in [(6, 0, 0), (3, 4, 3)]:
    _n = 4 ** (_pat[0] + _pat[2])
    _tbl = _rc_code_table(_pat)
    _folded = np.minimum(np.arange(_n), _tbl)
    _pal = int((_tbl == np.arange(_n)).sum())
    test_eq(len(set(_folded.tolist())), (_n + _pal) // 2)

rc table == revcomp over every code of [(1, 0, 1), (5, 0, 0), (6, 0, 0), (3, 4, 3), (4, 2, 4)]


## Scanning one pattern

A pattern is scored from two arrays: which motifs each sequence carries, and the totals of each
motif over its carriers. The library below is twelve inserts of twelve bases with `GAGACC`
(`GGTCTC` on the other strand) planted in four of them, and those four are the depleted ones, so
a correct scan must report `GAGACC` on four plasmids, all depleted.

In [ ]:
demo_lib = ["GAGACCTTAACG", "TTGAGACCGCAT", "ACGGTCTCAATG", "CCAAGAGACCTT",
            "ACGTTGCAATGC", "TTACGCATTGCA", "GCATTACGGTTA", "ATGCCATTGCAA",
            "CATGCAATTGCG", "TTGCAATGCCAT", "GCAATTGCATGC", "ATTGCCATGCAA"]
demo_fc = np.array([-2.5, -3.0, -3.5, -3.0, 0.2, -0.1, 0.0, 0.3, -0.2, 0.1, 0.0, -0.3])

pd.DataFrame({"insert": demo_lib, "log2FC": demo_fc,
              "carries the site": [("GAGACC" in s) or ("GGTCTC" in s) for s in demo_lib],
              "depleted (< -1)": demo_fc < -1})

,insert,log2FC,carries the site,depleted (< -1)
0,GAGACCTTAACG,-2.5,True,True
1,TTGAGACCGCAT,-3.0,True,True
2,ACGGTCTCAATG,-3.5,True,True
3,CCAAGAGACCTT,-3.0,True,True
4,ACGTTGCAATGC,0.2,False,False
5,TTACGCATTGCA,-0.1,False,False
6,GCATTACGGTTA,0.0,False,False
7,ATGCCATTGCAA,0.3,False,False
8,CATGCAATTGCG,-0.2,False,False
9,TTGCAATGCCAT,0.1,False,False


**Which motifs each sequence carries.** `_motif_occurrences` scans every window of both strands
and reduces the result to one entry per sequence and motif: a motif occurring twice in an insert,
or once on each strand, is one plasmid carrying it. For a self-rc pattern the two spellings of a
site are folded onto the smaller code (`_rc_code_table`); for any other pattern the rc spelling is
not in this code space, so there is nothing to fold. Either way each pattern emits one row per
plasmid per site.

In [ ]:
#| export
def _motif_occurrences(encoded_library, pattern):
    """Which motifs each sequence contains, as parallel `(sequence index, motif code)` arrays.

    Both strands are scanned and the result is reduced to one entry per sequence per motif, so
    a motif occurring several times in one sequence -- or once on each strand -- is counted
    once. Where a pattern is self-reverse-complementary the two spellings of a site are folded
    onto one code first, so `GAGACC` and `GGTCTC` come back as a single motif rather than two
    rows carrying identical statistics. Windows containing a non-ACGT base, or running off the
    end of a short sequence, are dropped: both carry the out-of-range code that
    `_pattern_window_codes` assigns.

    Returns:
        (seq_idx, motif_codes): equal-length int arrays. `seq_idx` is non-decreasing, so the
        pairs come out grouped by sequence.
    """
    fwd, rc = encoded_library
    n_codes = 4 ** (pattern[0] + pattern[2])
    codes = np.concatenate([_pattern_window_codes(fwd, pattern),
                            _pattern_window_codes(rc, pattern)], axis=1)
    if _pattern_is_self_rc(pattern):
        # One site, two spellings, both in this code space -- fold each onto the smaller of the
        # pair. Rejected windows keep their out-of-range code: they are not in the table, so
        # they are held out of the lookup and passed through untouched.
        table = _rc_code_table(pattern)
        inside = codes < n_codes
        codes = np.where(inside, np.minimum(codes, table[np.where(inside, codes, 0)]), codes)
    codes.sort(axis=1)                                  # repeats become adjacent
    keep = codes < n_codes                              # drop rejected windows
    keep[:, 1:] &= codes[:, 1:] != codes[:, :-1]        # drop repeats within a sequence
    return np.nonzero(keep)[0], codes[keep]

In [ ]:
seq_idx, motif_codes = _motif_occurrences(encode_library(demo_lib), (6, 0, 0))
occurrences = pd.DataFrame({
    "sequence": [demo_lib[i] for i in seq_idx],
    "motif": _decode_motif_codes(motif_codes, (6, 0, 0)),
    # the code IS the motif: A=0 C=1 G=2 T=3, read as a base-4 numeral
    "code (base 4)": [np.base_repr(c, 4).zfill(6) for c in motif_codes],
    "code": motif_codes})
print(f"{len(occurrences)} (sequence, motif) pairs from {len(demo_lib)} sequences")
occurrences.head(8)

82 (sequence, motif) pairs from 12 sequences


,sequence,motif,code (base 4),code
0,GAGACCTTAACG,AAGGTC,002231,173
1,GAGACCTTAACG,ACCTTA,011330,380
2,GAGACCTTAACG,AGACCT,020113,535
3,GAGACCTTAACG,CCTTAA,113300,1520
4,GAGACCTTAACG,CGTTAA,123300,1776
5,GAGACCTTAACG,CTTAAC,133001,1985
6,GAGACCTTAACG,GAGACC,202011,2181
7,TTGAGACCGCAT,ACCGCA,011210,356


Each row is one plasmid carrying one motif, and the base-4 column shows that a code **is** the
motif: `GAGACC` is `202011`. The planted site:

In [ ]:
occurrences[occurrences.motif.isin(["GAGACC", "GGTCTC"])]

,sequence,motif,code (base 4),code
6,GAGACCTTAACG,GAGACC,202011,2181
11,TTGAGACCGCAT,GAGACC,202011,2181
18,ACGGTCTCAATG,GAGACC,202011,2181
27,CCAAGAGACCTT,GAGACC,202011,2181


Four sequences, one row each: `GGTCTC` is the same site read on the other strand, folded onto
`GAGACC`, the smaller numeral. Reporting re-spells it on the strand REBASE uses
(`canonical_motif`).

A window containing a base other than ACGT is dropped, and so is one running past the end of a
short sequence, since padding carries the same code as `N`:

In [ ]:
edge = ["GAGACCTT",           # clean
        "GAGACCTTNAAAAAA",    # an N at position 8, then six more bases
        "GAGAC"]              # shorter than the 6-base pattern
_s, _c = _motif_occurrences(encode_library(edge), (6, 0, 0))
pd.DataFrame({"sequence": [edge[i] for i in _s], "motif": _decode_motif_codes(_c, (6, 0, 0))})

,sequence,motif
0,GAGACCTT,AAGGTC
1,GAGACCTT,AGACCT
2,GAGACCTT,GAGACC
3,GAGACCTTNAAAAAA,AAAAAA
4,GAGACCTTNAAAAAA,AAGGTC
5,GAGACCTTNAAAAAA,AGACCT
6,GAGACCTTNAAAAAA,GAGACC


The second insert keeps the windows that end before its `N`, loses the six that cover it, and picks
up `AAAAAA` after it (`TTTTTT` on the other strand, folded). Every window of the third runs into
the padding.

In [ ]:
#| hide
_by = {e: sorted(m for i, m in zip(_s.tolist(), _decode_motif_codes(_c, (6, 0, 0))) if edge[i] == e)
       for e in edge}
test_eq(_by["GAGACCTT"], ["AAGGTC", "AGACCT", "GAGACC"])
test_eq(_by["GAGACCTTNAAAAAA"], ["AAAAAA", "AAGGTC", "AGACCT", "GAGACC"])   # N windows dropped
test_eq(_by["GAGAC"], [])                                                   # too short: none
test_eq(len(_motif_occurrences(encode_library(["GAGNCC"]), (6, 0, 0))[0]), 0)  # N anywhere kills it

**Totals per motif.** Every statistic the scan reports is a sum over a motif's carriers, so each is
one `np.bincount`: carriers, depleted carriers, the sum of their log2FC, the sum of their
background (each plasmid's expected depletion rate, from `plasmid_background`; here one rate for
all), and the first carrier, which makes the output order a function of the library rather than of
hash order.

In [ ]:
#| export
def _motif_totals(seq_idx, motif_codes, fc_values, background, log2FC_thr, n_codes):
    """Per motif code: sequences carrying it, how many of those are depleted, the sum of their
    log2FC, the sum of their background, and the index of the first sequence that carries it.

    Sums rather than means, because `np.bincount` adds; the means are taken once at the end.
    """
    seq_fc = fc_values[seq_idx]
    n_seqs = np.bincount(motif_codes, minlength=n_codes)
    n_depleted = np.bincount(motif_codes, weights=seq_fc < log2FC_thr, minlength=n_codes)
    fc_sum = np.bincount(motif_codes, weights=seq_fc, minlength=n_codes)
    bg_sum = np.bincount(motif_codes, weights=background[seq_idx], minlength=n_codes)
    first_seq = np.full(n_codes, len(fc_values))
    np.minimum.at(first_seq, motif_codes, seq_idx)
    return n_seqs, n_depleted, fc_sum, bg_sum, first_seq

In [ ]:
demo_bg = np.full(len(demo_lib), (demo_fc < -1).mean())      # one rate for all: 4 of 12
n_seqs, n_depleted, fc_sum, bg_sum, first_seq = _motif_totals(
    seq_idx, motif_codes, demo_fc, demo_bg, -1, 4 ** 6)

present = np.flatnonzero(n_seqs)                 # the motifs this library actually contains
totals = pd.DataFrame({"motif": _decode_motif_codes(present, (6, 0, 0)),
                       "n_seqs": n_seqs[present], "n_depleted": n_depleted[present],
                       "sum_log2fc": fc_sum[present], "sum_background": bg_sum[present],
                       "first_seq": first_seq[present]})
print(f"{len(totals)} motifs present out of {4 ** 6} possible")
totals.sort_values(["n_depleted", "n_seqs"], ascending=False).head(6)

53 motifs present out of 4096 possible


,motif,n_seqs,n_depleted,sum_log2fc,sum_background,first_seq
43,GAGACC,4,4.0,-12.0,1.333333,0
2,AAGGTC,2,2.0,-5.5,0.666667,0
13,AGACCG,2,2.0,-6.5,0.666667,1
14,AGACCT,2,2.0,-5.5,0.666667,0
47,GTCTCA,2,2.0,-6.5,0.666667,1
50,TCTCAA,2,2.0,-6.5,0.666667,1


`GAGACC` tops the table with four carriers, all depleted. `05_fast` turns these totals into the
reported scores, and adds a compiled kernel that computes them in one pass.

In [ ]:
#| hide
# _motif_occurrences: one entry per sequence per motif, however often it occurs or on whichever strand.
_e = encode_library(["GAGACCGAGACC", "TTTTTTTT"])          # GAGACC occurs TWICE in the first
_s, _c = _motif_occurrences(_e, (6, 0, 0))
_pairs = list(zip(_s.tolist(), _decode_motif_codes(_c, (6, 0, 0))))
test_eq(len(_pairs), len(set(_pairs)))                     # no (sequence, motif) pair repeats
test_eq(sum(1 for s, m in _pairs if s == 0 and m == "GAGACC"), 1)   # twice, both strands -> one
test_eq(sum(1 for s, m in _pairs if s == 0 and m == "GGTCTC"), 0)   # the rc spelling is folded
test_eq(sorted(m for s, m in _pairs if s == 1), ["AAAAAA"])         # TTTTTT folds onto AAAAAA
test_eq(len(_motif_occurrences(encode_library(["NNNNNNNN"]), (6, 0, 0))[0]), 0)

# An asymmetric pattern has nothing to fold, and is left exactly as it was: the rc of a
# (4, 4, 3) motif is a (3, 4, 4) motif, which is not in this code space at all.
_sa, _ca = _motif_occurrences(encode_library(["AAATACCCCCGTGAAA"]), (4, 4, 3))
test_eq(sum(1 for m in _decode_motif_codes(_ca, (4, 4, 3)) if m == "ATACNNNNGTG"), 1)

# _motif_totals: the totals are sums over exactly those pairs, each sequence's background included.
_n, _d, _f, _b, _first = _motif_totals(_s, _c, np.array([-5.0, 0.0]), np.array([0.25, 0.5]), -1, 4 ** 6)
_g = _encode_motif("GAGACC", (6, 0, 0))
test_eq((int(_n[_g]), int(_d[_g]), float(_f[_g]), float(_b[_g]), int(_first[_g])), (1, 1, -5.0, 0.25, 0))
_ta = _encode_motif("AAAAAA", (6, 0, 0))                    # the folded home of TTTTTT
test_eq((int(_n[_ta]), int(_d[_ta]), float(_b[_ta]), int(_first[_ta])), (1, 0, 0.5, 1))
test_eq(int(_n[_encode_motif("CCCCCC", (6, 0, 0))]), 0)     # absent -> no sequences

## Reading motif presence off the window codes

Scoring a pattern asks what motifs a sequence contains. The rest of the pipeline asks the reverse,
of a handful of specific candidates: **which sequences contain this motif, on either strand?**

Done directly that is a regex per sequence per motif, and a noisy strain, which carries the most
candidates, pays for it most. Instead presence is a **lookup in the window codes**, done once per
pattern rather than once per motif: for 200 motifs of 4 patterns on the 150 nt example library
(`counts_panel.csv.gz`, 30,108 inserts after the count filter), 0.3 s against 6 s for the regex, with identical answers.

`_motif_presence_matrix` returns `(matrix, handled)`. A motif the fast path cannot encode —
IUPAC-degenerate, or of an unknown pattern — is left as an all-`False` column and flagged in
`handled`, so the caller does not read a blank column as "absent": `core` reads a degenerate motif
through its instances, each of which has a code, and a motif of unknown pattern by regex.

In [ ]:
#| export
def _motif_presence_matrix(encoded_library, motifs, patterns):
    """Boolean (n_sequences, n_motifs): does sequence i contain motif j, on either strand?

    One pass per distinct pattern rather than one per motif, and a lookup into the window codes
    rather than a regex per sequence. Motifs whose pattern is unknown, or which contain IUPAC
    ambiguity codes, are left as all-False here and not handled; `core` reads a degenerate motif
    through its instances (`_motif_instances`) and an unknown pattern by regex.

    Returns `(matrix, handled)` where `handled` marks the motifs the matrix actually covers.
    """
    fwd, rc = encoded_library
    n_seq = fwd.shape[0]
    matrix = np.zeros((n_seq, len(motifs)), dtype=bool)
    handled = np.zeros(len(motifs), dtype=bool)

    by_pattern = defaultdict(list)
    for j, (motif, pattern) in enumerate(zip(motifs, patterns)):
        if pattern is None:
            continue
        code = _encode_motif(motif, pattern)
        if code is not None:
            by_pattern[pattern].append((j, code))

    for pattern, entries in by_pattern.items():
        n_codes = 4 ** (pattern[0] + pattern[2])
        # Two motifs can share a code -- a site and its reverse complement, or the same site
        # listed twice -- so each distinct code gets a group, and every column of the group is
        # filled from it.
        groups = defaultdict(list)
        for j, code in entries:
            groups[code].append(j)
        lookup = np.full(n_codes + 1, -1, dtype=np.int32)      # -1: codes we do not care about
        for g, code in enumerate(groups):
            lookup[code] = g
        codes = np.concatenate(
            [_pattern_window_codes(fwd, pattern), _pattern_window_codes(rc, pattern)], axis=1)
        hit = lookup[codes]
        rows, where = np.nonzero(hit >= 0)
        present = np.zeros((n_seq, len(groups)), dtype=bool)
        present[rows, hit[rows, where]] = True
        for g, cols in enumerate(groups.values()):
            matrix[:, cols] = present[:, [g]]
        for j, _ in entries:
            handled[j] = True

    return matrix, handled

In [ ]:
_seqs = ["GAGACCTT", "AAGGTCTC", "TTTTTTTT", "GGGGGGGG"]
_m, _handled = _motif_presence_matrix(encode_library(_seqs), ["GAGACC", "TTTTTT"],
                                     [(6, 0, 0), (6, 0, 0)])
pd.DataFrame(_m, index=_seqs, columns=["GAGACC", "TTTTTT"])

,GAGACC,TTTTTT
GAGACCTT,True,False
AAGGTCTC,True,False
TTTTTTTT,False,True
GGGGGGGG,False,False


In [ ]:
#| hide
# GAGACC is in the first two sequences (the second spells it GGTCTC), TTTTTT only in the third; GGGGGGGG carries neither
test_eq(_m.tolist(), [[True, False], [True, False], [False, True], [False, False]])
test_eq(_handled.tolist(), [True, True])     # both are plain ACGT motifs: the matrix covers them

# A site listed twice, or with its reverse complement, fills every one of its columns: both
# spellings name the same plasmids.
_m, _h = _motif_presence_matrix(encode_library(["AAGGTCTCAA", "AAGAGACCAA", "CCCCCCCCCC"]),
                               ["GGTCTC", "GAGACC", "GGTCTC"], [(6, 0, 0)] * 3)
test_eq(_m.T.tolist(), [[True, True, False]] * 3)

One row per sequence, one column per motif. `GAGACC` is found in the second sequence only on the
other strand, where it is spelled `GGTCTC`: presence means "on either strand".

In [ ]:
#| hide
_enc = encode_library(["GAGACCTT", "AAGAGACC", "TTTTTTTT"])
_m, _h = _motif_presence_matrix(_enc, ["GAGACC", "TTTTTT"], [(6, 0, 0), (6, 0, 0)])
test_eq(_h.tolist(), [True, True])
test_eq(_m[:, 0].tolist(), [True, True, False])
test_eq(_m[:, 1].tolist(), [False, False, True])
# both strands are searched, so the reverse complement finds the same sequences
_m2, _ = _motif_presence_matrix(_enc, ["GGTCTC"], [(6, 0, 0)])
test_eq(_m2[:, 0].tolist(), [True, True, False])
# what the fast path cannot encode is reported as unhandled and left False, NOT as absent --
# the caller reads it another way, so a silent wrong answer is impossible
_m3, _h3 = _motif_presence_matrix(_enc, ["GAGAYC"], [(6, 0, 0)])
test_eq(_h3.tolist(), [False])
test_eq(int(_m3.sum()), 0)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
#| hide